In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pickle
import pandas as pd
from skimage.color import rgb2hed, hed2rgb
from skimage import exposure
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import torch
import einops

In [ ]:
which_block = "SU-24-17445.B1"

block_meta = pd.read_csv(f"/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/{which_block}_blockmeta.csv")
rgb = np.expand_dims(block_meta[["pixel_mean_r","pixel_mean_g","pixel_mean_b"]], axis=0).astype(np.uint8)
hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
hsv[:,:,1] = np.maximum(hsv[:,:,1], 64)
plt.figure()
plt.imshow(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

coords = np.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/SU-24-17445.B1_coords_256.npy")

xformed_coord = (coords/512).astype(int)
xformed_coord = xformed_coord - np.expand_dims(xformed_coord.min(axis=0), 0)
preview = np.zeros((xformed_coord.max(axis=0)+1))
preview[xformed_coord[:,0],xformed_coord[:,1]] = 1
plt.figure()
plt.imshow(preview)

#with open("/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/SU-18-72877.AFS1_coords_failed_256.pkl", "rb") as fd:
#    coords_fail = pickle.load(fd)

print((coords.shape[0], len(block_meta)))

block_meta

In [ ]:
for i in np.random.permutation(range(coords.shape[0]))[:40]:
    patch = get_patch_stain_sep(fd_full, i)
    patch = np.hstack([np.vstack(j) for j in patch])
    _, ax = plt.subplots(1, 1,figsize=(30,3))
    ax.imshow(patch)

In [ ]:
fd_full = np.memmap("/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/SU-24-17445.B1_patches_256.dat", dtype=np.uint8, mode="r", shape=(coords.shape[0], len(block_meta), 256,256,3))

In [ ]:
def get_patch_stain_sep(fd_full, i):
    images = fd_full[i, ...]

    patch = []
    for j in range(images.shape[0]):
        ihc_hed = rgb2hed(images[j])
        null = np.zeros_like(ihc_hed[:, :, 0])
        ihc_h = (hed2rgb(np.stack((ihc_hed[:, :, 0], null, null), axis=-1))*255).astype(np.uint8)
        ihc_e = (hed2rgb(np.stack((null, ihc_hed[:, :, 1], null), axis=-1))*255).astype(np.uint8)
        ihc_d = (hed2rgb(np.stack((null, null, ihc_hed[:, :, 2]), axis=-1))*255).astype(np.uint8)

        patch.append([images[j], ihc_h, ihc_d])
        #atch.append([images[j]/255, exposure.equalize_adapthist(ihc_h, clip_limit=0.03), ihc_d])
    return np.stack(patch)


In [ ]:
grid = np.stack([np.stack([coords[1010] + [j, i] for j in range(-2048,2048,512)]) for i in range(-2048,2048,512)])
points=coords

In [ ]:

dtype = [('x', grid.dtype), ('y', grid.dtype)]
structured_grid = grid.view(dtype).reshape(-1)  # Flattened structured version of grid
structured_points = points.view(dtype).reshape(-1)  # (N,) structured array

# Find indices where each grid coordinate matches points
indices = np.array([np.where(structured_points == g)[0][0] if g in structured_points else -1 for g in structured_grid])

# Reshape to (16,16) for index mapping
indices = indices.reshape(8,8)


In [ ]:
images = np.array([[get_patch_stain_sep(fd_full,j) if j>0 else np.ones((5,3, 256,256,3))*255 for j in i ] for i in indices])

In [ ]:
images.shape

In [ ]:
images2 = einops.rearrange(images, "bh bw s ss h w c -> ss s (bh h) (bw w) c")

In [ ]:
images2.shape

In [ ]:
for i,ssep in  zip(images2, ["orig", "counter", "dab"]):
    for j,st in zip(i, ["he","cd68", "gfap", "idh1", "ki67"]):
        Image.fromarray(j).save(f"SU-23-13946.B4_{st}_{ssep}.png")


In [ ]:
!mkdir SU-23-13946.B4_stain_sep
!mv SU-23-13946.B4* SU-23-13946.B4_stain_sep
!tar czvf

In [ ]:
plt.imshow((images2[1,0,...]).astype(np.uint8))

In [ ]:
plt.imshow((images2[1,1,...]*255).astype(np.uint8))

In [ ]:
images2[0,...]

In [ ]:
ihc_hed = rgb2hed(images2[0,...])
null = np.zeros_like(ihc_hed[:, :, 0])
ihc_h = hed2rgb(np.stack((ihc_hed[:, :, 0], null, null), axis=-1))#*255).astype(np.uint8)
ihc_e = hed2rgb(np.stack((null, ihc_hed[:, :, 1], null), axis=-1))#*255).astype(np.uint8)
ihc_d = hed2rgb(np.stack((null, null, ihc_hed[:, :, 2]), axis=-1))#*255).astype(np.uint8)

In [ ]:
plt.imshow(ihc_d)

In [ ]:
fd_full = np.memmap("/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/SU-23-13946.B4_patches_256.dat", dtype='uint8', mode='r', shape=(coords.shape[0],len(block_meta), 256, 256, 3))


In [ ]:
dinov2_processor = AutoImageProcessor.from_pretrained(
            "/nfs/mm-isilon/brainscans/dropbox/exp/models/dinov2/base")
dinov2_model = AutoModel.from_pretrained("/nfs/mm-isilon/brainscans/dropbox/exp/models/dinov2/base")


In [ ]:
fd_full.shape

In [ ]:
patch_viz = np.hstack([np.vstack(j) for j in patch])
_, ax = plt.subplots(1, 1,figsize=(30,30))
ax.imshow(patch_viz)

In [ ]:
raw_im = [
    Image.fromarray((i*255).astype(np.uint8)) for i in patch[:,1,...]
]

proc_im = torch.stack([
    dinov2_processor(ri, return_tensors="pt")["pixel_values"][0]
    for ri in raw_im
])
output = dinov2_model(pixel_values=proc_im)


In [ ]:
cls = output.last_hidden_state[:,0,:]
patch_emb = output.last_hidden_state[:,1:,:]

In [ ]:
patch_emb.shape

In [ ]:
patch_emb = torch.nn.functional.normalize(patch_emb, dim=2)
token_sim = einops.einsum(patch_emb, patch_emb, "s0 p0 d, s1 p1 d -> s0 p0 s1 p1").detach().numpy()
#token_sim = einops.rearrange(token_sim, "s0 p0 s1 p1 -> (s0 p0) (s1 p1)")
#token_sim = einops.reduce(token_sim, "s0 p0 s1 p1 -> s0 s1", "mean")


diag_block = np.eye(patch_emb.shape[1])
mask = einops.repeat(diag_block, "p1 p2 -> s1 p1 s2 p2", s1=patch_emb.shape[0], s2=patch_emb.shape[0])
token_sim = einops.reduce(token_sim*mask, "s0 p0 s1 p1 -> s0 s1", "sum") / patch_emb.shape[1]

plt.imshow(token_sim, vmin=0, vmax=1)

In [ ]:
token_sim.min()

In [ ]:
token_sim[8,:]

In [ ]:
cls_normed = torch.nn.functional.normalize(cls, dim=1)
cls_sim = cls_normed @ cls_normed.T
plt.imshow(cls_sim.detach().numpy(), vmin=0, vmax=1)

In [ ]:
cls_sim.min()

In [ ]:
for i in range(3000,3030):
    patch = get_patch_stain_sep(fd_full, i)
    patch = np.hstack([np.vstack(j) for j in patch])
    _, ax = plt.subplots(1, 1,figsize=(30,30))
    ax.imshow(patch)

In [ ]:
patch = fd_full[42, ...]

In [ ]:
_, ax = plt.subplots(1,1,figsize=(20,10))
ax.imshow(np.hstack(patch))

In [ ]:
from skimage import exposure

In [ ]:
for i in range(len(patch)):
    ihc_hed = rgb2hed(patch[i])
    null = np.zeros_like(ihc_hed[:, :, 0])
    ihc_h = hed2rgb(np.stack((ihc_hed[:, :, 0], null, null), axis=-1))
    ihc_e = hed2rgb(np.stack((null, ihc_hed[:, :, 1], null), axis=-1))
    ihc_d = hed2rgb(np.stack((null, null, ihc_hed[:, :, 2]), axis=-1))

    # Display
    fig, axes = plt.subplots(2, 2, figsize=(7, 6), sharex=True, sharey=True)
    ax = axes.ravel()

    ax[0].imshow(patch[i])
    ax[0].set_title("Original image")

    ax[1].imshow(exposure.adjust_gamma(ihc_h))
    ax[1].set_title("Hematoxylin")

    ax[2].imshow(ihc_e)
    ax[2].set_title("Eosin")  # Note that there is no Eosin stain in this image

    ax[3].imshow(ihc_d)
    ax[3].set_title("DAB")